# EDA METADATA

In [ ]:
target_columns = ['localization', 'dx']

for col in target_columns:
    print(f"--- Unique Values in Column: {col} ---")
    print(df[col].unique())
    print()

In [ ]:
print("Distribution of 'sex' variable:")
print(df['sex'].value_counts())

plt.figure(figsize=(8, 6))
sns.countplot(x='sex', data=df, palette='viridis')
plt.title('Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

In [ ]:
print("Distribution of 'age' variable:")
print(df['age'].value_counts())

plt.figure(figsize=(8, 6))
sns.countplot(x='age', data=df, palette='viridis')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df, x='dx',
                   order=df['dx'].value_counts().index,
                   palette='Set2')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.title('Distribution of Skin Cancer Classes - HAM10000', fontsize=14)
plt.xlabel('Lesion Type')
plt.ylabel('Sample Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.boxplot(data=df, x='dx', y='age',
                 order=df.groupby('dx')['age'].median().sort_values().index,
                 palette='Set2')
plt.title('Age Distribution per Class', fontsize=13, fontweight='bold')
plt.xlabel('Lesion Type', fontsize=10)
plt.ylabel('Age', fontsize=10)
plt.xticks(rotation=30, ha='right')
plt.yticks(np.arange(0, 81, 10))
plt.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)

# Add the median number above each box
medians = df.groupby('dx')['age'].median()
order = medians.sort_values().index
for i, dx in enumerate(order):
    median_val = medians[dx]
    ax.text(i, median_val + 1, f'{median_val:.0f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='localization', order=df['localization'].value_counts().index, palette='viridis')
plt.title('Lesion Localization Distribution')
plt.xlabel('Count')
plt.ylabel('Lesion Localization')
plt.tight_layout()
plt.show()

localization_counts = df['localization'].value_counts().reset_index()
localization_counts.columns = ['Lesion Localization', 'Sample Count']
display(localization_counts)

In [ ]:
pivot = df.groupby(['dx', 'localization']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Skin Cancer Class vs. Lesion Localization Distribution')
plt.xlabel('Localization')
plt.ylabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
# Age binning
df['age_group'] = pd.cut(df['age'], bins=[0, 20, 40, 60, 80, 100],
                          labels=['<20', '20-40', '40-60', '60-80', '>80'])

# Crosstab: Class vs Age Group
ct_age = pd.crosstab(df['dx'], df['age_group'])
plt.figure(figsize=(10, 5))
ct_age.plot(kind='bar', figsize=(12, 5), colormap='Set2')
plt.title('Skin Cancer Class Distribution per Age Group')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Age Group')
plt.tight_layout()
plt.show()

# Crosstab: Class vs Gender
ct_sex = pd.crosstab(df['dx'], df['sex'], normalize='index') * 100
plt.figure(figsize=(14, 5))
ct_sex.plot(kind='bar', figsize=(12, 5), colormap='coolwarm')
plt.title('Gender Proportion per Skin Cancer Class (%)')
plt.xlabel('Class')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Overall Distribution (Top Plot)
sns.countplot(data=df, x='dx_type', palette='Set3', ax=axes[0])
axes[0].set_title('Diagnosis Method Distribution', fontsize=12, pad=10)
axes[0].set_xlabel('Method')
axes[0].set_ylabel('Count')

# Add numeric annotations on top of the first plot's bars
for p in axes[0].patches:
    height = p.get_height()
    if height > 0:  # Only display if height > 0
        axes[0].annotate(f'{int(height)}',
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                         textcoords='offset points')

# Diagnosis Method per Class (Bottom Plot)
ct_type = pd.crosstab(df['dx'], df['dx_type'], normalize='index') * 100
ct_type.plot(kind='bar', ax=axes[1], colormap='Set3')
axes[1].set_title('Proportion of Diagnosis Methods per Class (%)', fontsize=12, pad=10)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Method', fontsize=8)

# Add percentage annotations on top of the second plot's bars
for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.1f}%',
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', fontsize=8, xytext=(0, 3),
                         textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# 1. TOP PLOT: Patient Age Distribution by Diagnosis Method (Boxplot)
sns.boxplot(data=df, x='dx_type', y='age', palette='Pastel1', ax=axes[0])
axes[0].set_title('Patient Age Distribution by Diagnosis Method', fontsize=14, pad=15)
axes[0].set_xlabel('Diagnosis Method (dx_type)', fontsize=12)
axes[0].set_ylabel('Age (Years)', fontsize=12)

# Add numeric values (Mean age) inside the boxplot
mean_ages_type = df.groupby('dx_type')['age'].mean()
for i, type_class in enumerate(mean_ages_type.index):
    axes[0].text(i, mean_ages_type[type_class], f'Mean:\n{mean_ages_type[type_class]:.1f}',
                 ha='center', va='center', color='black', weight='bold', fontsize=9,
                 bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.3'))


# 2. BOTTOM PLOT: Gender Proportion by Diagnosis Method (Bar Chart)
ct_sex_type = pd.crosstab(df['dx_type'], df['sex'], normalize='index') * 100
ct_sex_type.plot(kind='bar', ax=axes[1], colormap='Pastel1')
axes[1].set_title('Gender Proportion by Diagnosis Method (%)', fontsize=14, pad=15)
axes[1].set_xlabel('Diagnosis Method (dx_type)', fontsize=12)
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Gender', fontsize=10)

# Add percentage annotations on top of each bar
for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.1f}%',
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                         textcoords='offset points')

plt.tight_layout()
plt.show()

# IMAGE EDA HAM1000

In [ ]:
COLS = 5
N = 10

for dx_folder_name in os.listdir(image_data_dir):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)

    if not os.path.isdir(dx_folder_path):
        continue

    images = [f for f in os.listdir(dx_folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if not images:
        print(f"No images in folder '{dx_folder_name}'.")
        continue

    selected = random.sample(images, min(N, len(images)))
    rows = (len(selected) + COLS - 1) // COLS

    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3, rows * 3))
    axes = axes.flatten()

    for i, fname in enumerate(selected):
        axes[i].imshow(Image.open(os.path.join(dx_folder_path, fname)))
        axes[i].axis('off')

    for ax in axes[len(selected):]:
        ax.axis('off')

    plt.suptitle(f'Sample {N} Images from {dx_folder_name}', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
def get_image_hash(img_path):
    with open(img_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Collect all hashes along with their paths
hash_map = defaultdict(list)

for dx_folder_name in os.listdir(image_data_dir):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)

    if not os.path.isdir(dx_folder_path):
        continue

    for fname in os.listdir(dx_folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(dx_folder_path, fname)
            img_hash = get_image_hash(img_path)
            hash_map[img_hash].append(img_path)

# Filter duplicates only
duplicates = {h: paths for h, paths in hash_map.items() if len(paths) > 1}

# Display results
total_images = sum(len(p) for p in hash_map.values())
print(f"Total images checked : {total_images}")
print(f"Duplicates found     : {len(duplicates)} groups\n")

if duplicates:
    for i, (h, paths) in enumerate(duplicates.items(), 1):
        print(f"Duplicate #{i} (hash: {h[:10]}...):")
        for p in paths:
            print(f"  - {p}")
        print()
else:
    print("No duplicates found.")

In [ ]:
def get_image_hash(img_path):
    with open(img_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = defaultdict(list)

for dx_folder_name in os.listdir(image_data_dir):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)
    if not os.path.isdir(dx_folder_path):
        continue
    for fname in os.listdir(dx_folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(dx_folder_path, fname)
            hash_map[get_image_hash(img_path)].append(img_path)

duplicates = {h: paths for h, paths in hash_map.items() if len(paths) > 1}

print(f"Duplicates found: {len(duplicates)} groups\n")

if duplicates:
    for i, (h, paths) in enumerate(duplicates.items(), 1):
        n = len(paths)
        fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
        fig.patch.set_facecolor('#f8f8f8')

        if n == 1:
            axes = [axes]

        for ax, img_path in zip(axes, paths):
            img = Image.open(img_path).copy()
            ax.imshow(img)
            ax.set_title(f"{os.path.basename(os.path.dirname(img_path))}\n{os.path.basename(img_path)}", fontsize=8)
            ax.axis('off')

        # Translate plot title parameters
        plt.suptitle(f'Duplicate #{i}  |  hash: {h[:10]}...  |  {n} files', fontsize=11, fontweight='bold', y=1.05)
        plt.tight_layout(pad=1.5)
        plt.show()
        print()
else:
    print("No duplicates found.")

In [ ]:
# Check imbalance across labels
class_counts = {d: len(os.listdir(os.path.join(image_data_dir, d)))
                for d in os.listdir(image_data_dir)
                if os.path.isdir(os.path.join(image_data_dir, d))}

plt.figure(figsize=(10, 5))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.xticks(rotation=15, ha='right')
plt.title('Image Distribution per Class')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

In [ ]:
# Check if all images have uniform sizes
sizes = []
for dx_folder_name in os.listdir(image_data_dir):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)
    if not os.path.isdir(dx_folder_path):
        continue
    for fname in os.listdir(dx_folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            with Image.open(os.path.join(dx_folder_path, fname)) as img:
                sizes.append({'label': dx_folder_name, 'width': img.size[0], 'height': img.size[1], 'mode': img.mode})

df_sizes = pd.DataFrame(sizes)
print(df_sizes[['width', 'height']].describe())
print(f"\nUnique sizes  : {df_sizes[['width', 'height']].drop_duplicates().shape[0]}")
print(f"Unique modes  : {df_sizes['mode'].unique()}")

In [ ]:
# Check average brightness per class
brightness = []
for dx_folder_name in os.listdir(image_data_dir):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)
    if not os.path.isdir(dx_folder_path):
        continue
    for fname in random.sample(os.listdir(dx_folder_path), min(50, len(os.listdir(dx_folder_path)))):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img = np.array(Image.open(os.path.join(dx_folder_path, fname)).convert('RGB'))
            brightness.append({'label': dx_folder_name, 'mean': img.mean(), 'std': img.std()})

df_bright = pd.DataFrame(brightness)
# Plot setting
ax = df_bright.groupby('label')[['mean', 'std']].mean().plot(kind='bar', figsize=(10, 5))
plt.title('Average Brightness & Contrast per Class')
plt.xlabel('Class Label')
plt.ylabel('Value')
plt.legend(['Brightness (Mean)', 'Contrast (Std)'])
plt.tight_layout()
plt.show()

In [ ]:
# Check color distribution per class (sample)
fig, axes = plt.subplots(len(class_counts), 3, figsize=(12, len(class_counts) * 2))
colors = ['red', 'green', 'blue']

for row, dx_folder_name in enumerate(class_counts.keys()):
    dx_folder_path = os.path.join(image_data_dir, dx_folder_name)
    sample_imgs = random.sample([f for f in os.listdir(dx_folder_path)
                                  if f.lower().endswith(('.jpg','.jpeg','.png'))], min(10, len(os.listdir(dx_folder_path))))
    pixels = np.concatenate([np.array(Image.open(os.path.join(dx_folder_path, f)).convert('RGB'))
                              for f in sample_imgs], axis=0).reshape(-1, 3)

    for col, (ch, color) in enumerate(zip(range(3), colors)):
        axes[row, col].hist(pixels[:, ch], bins=50, color=color, alpha=0.7)
        # Capitalize the color name for the title
        axes[row, col].set_title(f'{dx_folder_name} - {color.capitalize()}', fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('RGB Distribution per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
augmentations = {
    'Original'        : transforms.Compose([transforms.Resize((224, 224))]),
    'Horizontal Flip' : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1)]),
    'Vertical Flip'   : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomVerticalFlip(p=1)]),
    'Rotation'        : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomRotation(30)]),
    'Color Jitter'    : transforms.Compose([transforms.Resize((224, 224)), transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)]),
    'Grayscale'       : transforms.Compose([transforms.Resize((224, 224)), transforms.Grayscale(num_output_channels=3)]),
}

# Get 1 sample image
sample_folder = os.path.join(extract_path, os.listdir(extract_path)[0])
sample_img_path = os.path.join(sample_folder,
                  [f for f in os.listdir(sample_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))][0])
img = Image.open(sample_img_path).convert('RGB')

# Display
fig, axes = plt.subplots(1, len(augmentations), figsize=(18, 4))
fig.patch.set_facecolor('#f8f9fa')

for ax, (title, transform) in zip(axes, augmentations.items()):
    aug_img = transform(img)
    ax.imshow(aug_img)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Image Augmentation Preview', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(pad=1.5)
plt.show()

In [ ]:
def get_mean_color(img_path, size=(64, 64)):
    img = Image.open(img_path).convert('RGB').resize(size)
    arr = np.array(img) / 255.0
    return arr.mean(axis=(0, 1))  # mean R, G, B

# Take a sample of 100 images per class
results = []
for dx in df['dx'].unique():
    samples = df[df['dx'] == dx].dropna(subset=['img_path']).sample(
                min(100, len(df[df['dx'] == dx])), random_state=42)
    for _, row in samples.iterrows():
        r, g, b = get_mean_color(row['img_path'])
        results.append({'dx': dx, 'R': r, 'G': g, 'B': b})

color_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, channel in enumerate(['R', 'G', 'B']):
    sns.boxplot(data=color_df, x='dx', y=channel,
                palette='Set2', ax=axes[i])
    axes[i].set_title(f'{channel} Channel Distribution per Class')
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel('Mean Pixel Value')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, (dx, name) in enumerate(label_map.items()):
    samples = df[df['dx'] == dx].dropna(subset=['img_path']).sample(
                min(50, len(df[df['dx'] == dx])), random_state=42)

    all_r, all_g, all_b = [], [], []
    for _, row in samples.iterrows():
        img = np.array(Image.open(row['img_path']).convert('RGB')) / 255.0
        all_r.extend(img[:,:,0].flatten())
        all_g.extend(img[:,:,1].flatten())
        all_b.extend(img[:,:,2].flatten())

    axes[i].hist(all_r, bins=50, alpha=0.6, color='red',   label='R', density=True)
    axes[i].hist(all_g, bins=50, alpha=0.6, color='green', label='G', density=True)
    axes[i].hist(all_b, bins=50, alpha=0.6, color='blue',  label='B', density=True)
    axes[i].set_title(f'{dx.upper()} - {name}', fontsize=9)
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=7)

axes[-1].axis('off')
plt.suptitle('Pixel Histogram per Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
IMG_SIZE = (128, 128)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (dx, name) in enumerate(label_map.items()):
    samples = df[df['dx'] == dx].dropna(subset=['img_path']).sample(
                min(100, len(df[df['dx'] == dx])), random_state=42)

    avg_img = np.zeros((*IMG_SIZE, 3), dtype=np.float64)
    for _, row in samples.iterrows():
        img = np.array(Image.open(row['img_path']).convert('RGB').resize(IMG_SIZE))
        avg_img += img / 255.0

    avg_img /= len(samples)

    axes[i].imshow(avg_img)
    axes[i].set_title(f'{dx.upper()} | {name}', fontsize=9)
    axes[i].axis('off')

axes[-1].axis('off')
plt.suptitle('Average Image per Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def variance_of_laplacian(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

# Calculate blur score per class (sample of 100 images)
blur_results = []
for dx in df['dx'].unique():
    samples = df[df['dx'] == dx].dropna(subset=['img_path']).sample(
                min(100, len(df[df['dx'] == dx])), random_state=42)
    for _, row in samples.iterrows():
        score = variance_of_laplacian(row['img_path'])
        blur_results.append({'dx': dx, 'blur_score': score})

blur_df = pd.DataFrame(blur_results)

plt.figure(figsize=(10, 5))
sns.boxplot(data=blur_df, x='dx', y='blur_score', palette='Set2')
plt.title('Blur Score Distribution per Class (Higher = Sharper)')
plt.xlabel('Class')
plt.ylabel('Variance of Laplacian')
plt.tight_layout()
plt.show()

print("Average blur score per class:")
print(blur_df.groupby('dx')['blur_score'].mean().sort_values(ascending=False))

In [ ]:
def extract_glcm_features(img_path):
    img = Image.open(img_path).convert('L').resize((128, 128))
    arr = np.array(img)
    glcm = graycomatrix(arr, distances=[1], angles=[0], levels=256,
                        symmetric=True, normed=True)
    return {
        'contrast'     : graycoprops(glcm, 'contrast')[0, 0],
        'dissimilarity': graycoprops(glcm, 'dissimilarity')[0, 0],
        'homogeneity'  : graycoprops(glcm, 'homogeneity')[0, 0],
        'energy'       : graycoprops(glcm, 'energy')[0, 0],
        'correlation'  : graycoprops(glcm, 'correlation')[0, 0]
    }

# Extract GLCM features (sample of 50 images per class)
glcm_results = []
for dx in df['dx'].unique():
    samples = df[df['dx'] == dx].dropna(subset=['img_path']).sample(
                min(50, len(df[df['dx'] == dx])), random_state=42)
    for _, row in samples.iterrows():
        feats = extract_glcm_features(row['img_path'])
        feats['dx'] = dx
        glcm_results.append(feats)

glcm_df = pd.DataFrame(glcm_results)

# Visualize GLCM features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
features = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']

for i, feat in enumerate(features):
    sns.boxplot(data=glcm_df, x='dx', y=feat, palette='Set2', ax=axes[i])
    axes[i].set_title(f'GLCM - {feat.capitalize()}')
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel(feat)

axes[-1].axis('off')
plt.suptitle('GLCM Texture Analysis per Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
lesion_counts = df.groupby('lesion_id').size().reset_index(name='image_count')

print("Distribution of image counts per lesion/patient:")
print(lesion_counts['image_count'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
sns.countplot(data=lesion_counts, x='image_count', palette='Set2')
plt.title('Distribution of Image Counts per Lesion ID')
plt.xlabel('Number of Images per Lesion')
plt.ylabel('Number of Lesions')
plt.tight_layout()
plt.show()

print(f"\nTotal unique lesions     : {lesion_counts['lesion_id'].nunique()}")
print(f"Total images             : {len(df)}")
print(f"Lesions with >1 image    : {(lesion_counts['image_count'] > 1).sum()}")

In [ ]:
# Find the lesion_id with the highest number of images
top_lesion = lesion_counts.sort_values('image_count', ascending=False).iloc[0]['lesion_id']
dup_samples = df[df['lesion_id'] == top_lesion].dropna(subset=['img_path'])

print(f"Lesion ID        : {top_lesion}")
print(f"Class            : {dup_samples['dx'].values[0]}")
print(f"Number of photos : {len(dup_samples)}")

fig, axes = plt.subplots(1, len(dup_samples), figsize=(4 * len(dup_samples), 4))
if len(dup_samples) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, dup_samples.iterrows()):
    img = Image.open(row['img_path'])
    ax.imshow(img)
    ax.set_title(row['image_id'], fontsize=8)
    ax.axis('off')

plt.suptitle(f'Images from the Same Lesion/Patient (lesion_id: {top_lesion})', fontsize=12)
plt.tight_layout()
plt.show()

# EDA METADATA DERM12345

In [ ]:
print(merged['label'].unique())

In [ ]:
cols = ['image_type', 'super_class', 'malignancy', 'main_class_1', 'main_class_2', 'sub_class', 'label']

for col in cols:
    print(f"\n{col} ({merged[col].nunique()} unique):")
    print(merged[col].value_counts())

In [ ]:
# Cek placeholder seperti 'unknown', 'none', '-'
placeholders = ['unknown', 'none', '-', 'n/a', '']
print("\nPlaceholder values:")
for col in merged.columns:
    mask = merged[col].str.lower().isin(placeholders)
    if mask.sum() > 0:
        print(f"  {col}: {mask.sum()} placeholder")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 14))

# Bar Chart: Distribusi Label
label_counts = merged['label'].value_counts()
colors = plt.cm.Blues_r([i / len(label_counts) * 0.6 + 0.2 for i in range(len(label_counts))])

bars = axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribusi Label', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Jumlah Gambar', fontsize=11)
axes[0].set_xlabel('Label', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_facecolor('#f8f9fa')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].spines[['top', 'right']].set_visible(False)

# Tambah angka di atas bar yang besar saja
for bar in bars:
    if bar.get_height() > 200:
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                     f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

# Pie Chart: Distribusi Malignancy
malignancy_counts = merged['malignancy'].value_counts()
pie_colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']

wedges, texts, autotexts = axes[1].pie(
    malignancy_counts.values,
    labels=malignancy_counts.index,
    autopct='%1.1f%%',
    colors=pie_colors[:len(malignancy_counts)],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
for at in autotexts:
    at.set_fontweight('bold')

axes[1].set_title('Distribusi Malignancy', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout(pad=3)
plt.show()

In [ ]:
cols = ['super_class', 'main_class_1', 'main_class_2']
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

for ax, col in zip(axes, cols):
    counts = merged[col].value_counts()
    colors = plt.cm.Blues_r([i / len(counts) * 0.6 + 0.2 for i in range(len(counts))])

    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribusi {col}', fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('Jumlah Gambar', fontsize=10)
    ax.set_xlabel(col, fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.set_facecolor('#f8f9fa')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.spines[['top', 'right']].set_visible(False)

    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout(pad=3)
plt.show()

In [ ]:
print(merged.groupby(['super_class', 'main_class_1'])['main_class_2'].nunique())
print()
print(merged.groupby(['main_class_1', 'main_class_2'])['label'].nunique())

In [ ]:
ct = pd.crosstab(merged['super_class'], merged['malignancy'])

ax = ct.plot(kind='bar', figsize=(12, 6), colormap='Set2', edgecolor='white')
plt.title('Malignancy per Super Class', fontsize=13, fontweight='bold')
plt.ylabel('Jumlah')
plt.xticks(rotation=20)
plt.legend(title='Malignancy')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.gca().spines[['top', 'right']].set_visible(False)

# Tambah angka di atas setiap bar
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=8, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
img_per_patient = merged['patient_id'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#f8f9fa')

# --- Histogram ---
axes[0].hist(img_per_patient.values, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribusi Jumlah Gambar per Pasien', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Jumlah Gambar', fontsize=10)
axes[0].set_ylabel('Jumlah Pasien', fontsize=10)
axes[0].set_facecolor('#f8f9fa')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].spines[['top', 'right']].set_visible(False)

# Tambah garis median & mean
axes[0].axvline(img_per_patient.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {img_per_patient.median():.0f}')
axes[0].axvline(img_per_patient.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {img_per_patient.mean():.1f}')
axes[0].legend(fontsize=9)

# --- Bar Chart Frekuensi ---
freq = img_per_patient.value_counts().sort_index()
axes[1].bar(freq.index, freq.values, color='steelblue', edgecolor='white', linewidth=0.5, width=0.8)
axes[1].set_title('Frekuensi Gambar per Pasien', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Jumlah Gambar', fontsize=10)
axes[1].set_ylabel('Jumlah Pasien', fontsize=10)
axes[1].set_facecolor('#f8f9fa')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout(pad=3)
plt.show()

# Summary
print(f"Total pasien unik      : {merged['patient_id'].nunique():,}")
print(f"Rata-rata gambar/pasien: {img_per_patient.mean():.2f}")
print(f"Median gambar/pasien   : {img_per_patient.median():.0f}")
print(f"Maks gambar/pasien     : {img_per_patient.max()}")

In [ ]:
counts = merged['image_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(counts.index, counts.values, color=plt.cm.Set2.colors[:len(counts)], edgecolor='white')
ax.set_title('Distribusi Image Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(merged['image_type'], merged['super_class'], normalize='index') * 100
ct.plot(kind='bar', figsize=(12, 5), colormap='tab10', edgecolor='white')
plt.title('Proporsi Super Class per Image Type (%)', fontsize=13, fontweight='bold')
plt.ylabel('Proporsi (%)')
plt.xticks(rotation=15)
plt.legend(title='Super Class', bbox_to_anchor=(1.05, 1))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.gca().spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(merged['label'], merged['malignancy'], normalize='index') * 100

plt.figure(figsize=(10, 12))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.5,
            cbar_kws={'label': '%'})
plt.title('Proporsi Malignancy per Label (%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
counts = merged.groupby(['super_class', 'main_class_1']).size().reset_index(name='count')
plt.figure(figsize=(14, 7))
squarify.plot(sizes=counts['count'], label=counts['main_class_1'], alpha=0.8,
              color=plt.cm.Set3.colors[:len(counts)])
plt.title('Treemap: Distribusi main_class_1 dalam super_class', fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
patient_labels = merged.groupby('patient_id')['label'].nunique()
multi_label = patient_labels[patient_labels > 1]

print(f"Pasien dengan > 1 label berbeda: {len(multi_label)}")
print(f"\nDistribusi jumlah label per pasien:")
print(patient_labels.value_counts().sort_index())

# Contoh pasien dengan banyak label
if len(multi_label) > 0:
    sample_patient = multi_label.index[0]
    print(f"\nContoh pasien '{sample_patient}':")
    print(merged[merged['patient_id'] == sample_patient][['image_id', 'label', 'malignancy']])

In [ ]:
ct = pd.crosstab(merged['image_type'], merged['malignancy'], normalize='index') * 100
ax = ct.plot(kind='barh', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Rasio Malignancy per Image Type (%)', fontsize=13, fontweight='bold')
plt.xlabel('Proporsi (%)')
plt.legend(title='Malignancy')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.gca().spines[['top', 'right']].set_visible(False)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
top_patients = merged['patient_id'].value_counts().head(20).reset_index()
top_patients.columns = ['patient_id', 'count']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top_patients['patient_id'], top_patients['count'], color='steelblue', edgecolor='white')
ax.set_title('Top 20 Pasien dengan Gambar Terbanyak', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Gambar')
ax.tick_params(axis='x', rotation=45)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=8)
ax.set_facecolor('#f8f9fa')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# IMAGE EDA DERM12345

In [ ]:
sizes = []
for folder in os.listdir(extractpath):
    folder_path = os.path.join(extractpath, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            with Image.open(os.path.join(folder_path, fname)) as img:
                sizes.append({'label': folder, 'width': img.size[0], 'height': img.size[1], 'mode': img.mode})

df_sizes = pd.DataFrame(sizes)

print(df_sizes[['width', 'height']].describe())
print(f"\nUnique resolusi : {df_sizes[['width','height']].drop_duplicates().shape[0]}")
print(f"Unique mode     : {df_sizes['mode'].unique()}")

In [ ]:
unique_res = df_sizes[['width', 'height']].drop_duplicates().sort_values(['width', 'height'])
unique_res['resolution'] = unique_res['width'].astype(str) + 'x' + unique_res['height'].astype(str)
print(f"Total unique resolusi: {len(unique_res)}\n")
print(unique_res['resolution'].to_string(index=False))

In [ ]:
mode_counts = df_sizes['mode'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#f8f9fa')

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    mode_counts.values,
    labels=mode_counts.index,
    autopct='%1.1f%%',
    colors=['steelblue', '#e67e22'],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Proportion of RGB vs RGBA', fontsize=13, fontweight='bold')

# Bar chart
bars = axes[1].bar(mode_counts.index, mode_counts.values,
                   color=['steelblue', '#e67e22'], edgecolor='white', width=0.4)
axes[1].bar_label(bars, fmt='%d', padding=5, fontsize=10, fontweight='bold')
axes[1].set_title('Count of RGB vs RGBA', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Images')
axes[1].set_facecolor('#f8f9fa')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_ylim(0, mode_counts.max() * 1.15)

plt.suptitle('Image Mode Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(pad=3)
plt.show()

In [ ]:
TARGET_SIZE = (224, 224)
rgb_pixels  = []
rgba_pixels = []

for folder in os.listdir(extract_path):
    folder_path = os.path.join(extract_path, folder)
    if not os.path.isdir(folder_path):
        continue
    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for fname in random.sample(images, min(30, len(images))):
        img = Image.open(os.path.join(folder_path, fname))
        img_resized = img.resize(TARGET_SIZE)
        if img.mode == 'RGBA':
            rgba_pixels.append(np.array(img_resized))
        else:
            rgb_pixels.append(np.array(img_resized.convert('RGB')))

rgb_arr  = np.concatenate(rgb_pixels,  axis=0).reshape(-1, 3)
rgba_arr = np.concatenate(rgba_pixels, axis=0).reshape(-1, 4) if rgba_pixels else None

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.patch.set_facecolor('#f8f9fa')

# RGB channels
channel_info = [('R', 0, 'red'), ('G', 1, 'green'), ('B', 2, 'blue')]
for ch_name, ch_idx, color in channel_info:
    axes[0].hist(rgb_arr[:, ch_idx], bins=50, color=color, alpha=0.5, label=ch_name, edgecolor='none')
axes[0].set_title(f'RGB Channel Distribution ({len(rgb_pixels)} images)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Pixel Value (0-255)')
axes[0].set_ylabel('Frequency')
axes[0].legend(title='Channel')
axes[0].set_facecolor('#f8f9fa')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].spines[['top', 'right']].set_visible(False)

# RGBA channels
if rgba_arr is not None:
    channel_info_rgba = [('R', 0, 'red'), ('G', 1, 'green'), ('B', 2, 'blue'), ('A', 3, 'purple')]
    for ch_name, ch_idx, color in channel_info_rgba:
        axes[1].hist(rgba_arr[:, ch_idx], bins=50, color=color, alpha=0.5, label=ch_name, edgecolor='none')
    axes[1].set_title(f'RGBA Channel Distribution ({len(rgba_pixels)} images)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Pixel Value (0-255)')
    axes[1].set_ylabel('Frequency')
    axes[1].legend(title='Channel')
    axes[1].set_facecolor('#f8f9fa')
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)
    axes[1].spines[['top', 'right']].set_visible(False)
else:
    axes[1].text(0.5, 0.5, 'No RGBA images found',
                 ha='center', va='center', fontsize=12, color='gray')
    axes[1].axis('off')

plt.suptitle('Pixel Distribution per Channel — RGB vs RGBA', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(pad=3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 14))
fig.patch.set_facecolor('#f8f9fa')

for ax, col, title, color in zip(
    axes,
    ['width', 'height'],
    ['Average Width per Label', 'Average Height per Label'],
    ['steelblue', '#2ecc71']
):
    data = df_sizes.groupby('label')[col].mean().sort_values(ascending=True)
    bars = ax.barh(data.index, data.values, color=color, edgecolor='white', linewidth=0.5, height=0.7)

    # Values at the end of the bars
    for bar in bars:
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
                f'{int(bar.get_width()):,}px', va='center', fontsize=7.5)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(f'{col.capitalize()} (px)', fontsize=10)
    ax.set_ylabel('Label', fontsize=10)
    ax.tick_params(axis='y', labelsize=8.5)
    ax.set_facecolor('#f8f9fa')
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(0, data.max() * 1.18)  # Give room for text labels

plt.suptitle('Image Resolution Distribution per Label', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(pad=3)
plt.show()

In [ ]:
stats = []
for folder in os.listdir(extractpath):
    folder_path = os.path.join(extractpath, folder)
    if not os.path.isdir(folder_path):
        continue
    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for fname in random.sample(images, min(50, len(images))):
        img = np.array(Image.open(os.path.join(folder_path, fname)).convert('RGB'))
        stats.append({'label': folder, 'brightness': img.mean(), 'contrast': img.std()})

df_stats = pd.DataFrame(stats)

fig, axes = plt.subplots(2, 1, figsize=(14, 18))
fig.patch.set_facecolor('#f8f9fa')

for ax, col, title, color in zip(
    axes,
    ['brightness', 'contrast'],
    ['Brightness', 'Contrast'],
    ['steelblue', '#e67e22']
):
    data = df_stats.groupby('label')[col].mean().sort_values(ascending=True)
    bars = ax.barh(data.index, data.values, color=color, edgecolor='white', linewidth=0.5, height=0.7)

    # Values at the end of the bars
    for bar in bars:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{bar.get_width():.1f}', va='center', fontsize=8)

    ax.set_title(f'Average {title} per Label', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(title, fontsize=10)
    ax.set_ylabel('Label', fontsize=10)
    ax.tick_params(axis='y', labelsize=8.5)
    ax.set_facecolor('#f8f9fa')
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(0, data.max() * 1.15)

plt.suptitle('Brightness & Contrast Distribution per Label', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(pad=3)
plt.show()

In [ ]:
augmentations = {
    'Original'        : transforms.Compose([transforms.Resize((224, 224))]),
    'Horizontal Flip' : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1)]),
    'Vertical Flip'   : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomVerticalFlip(p=1)]),
    'Rotation'        : transforms.Compose([transforms.Resize((224, 224)), transforms.RandomRotation(30)]),
    'Color Jitter'    : transforms.Compose([transforms.Resize((224, 224)), transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)]),
    'Grayscale'       : transforms.Compose([transforms.Resize((224, 224)), transforms.Grayscale(num_output_channels=3)]),
}

# Get 1 sample image
sample_folder = os.path.join(extractpath, os.listdir(extractpath)[0])
sample_img_path = os.path.join(sample_folder,
                  [f for f in os.listdir(sample_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))][0])
img = Image.open(sample_img_path).convert('RGB')

# Display
fig, axes = plt.subplots(1, len(augmentations), figsize=(18, 4))
fig.patch.set_facecolor('#f8f9fa')

for ax, (title, transform) in zip(axes, augmentations.items()):
    aug_img = transform(img)
    ax.imshow(aug_img)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Image Augmentation Preview', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(pad=1.5)
plt.show()